In [1]:
!pip install transformers peft accelerate bitsandbytes trl datasets --quiet

print("✅ Packages installed")

✅ Packages installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from transformers import TrainingArguments, Trainer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config (correct way)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

print("✅ Model loaded successfully in 4-bit")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded successfully in 4-bit


In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA added")

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079
✅ LoRA added


In [4]:
technical_data = [
    {"instruction": "Explain how a transformer works simply.", "output": "Transformers use self-attention to process all tokens at once, capturing long-range dependencies better than RNNs."},
    {"instruction": "Difference between precision and recall?", "output": "Precision = accuracy of positive predictions. Recall = how many actual positives were found."},
    {"instruction": "What is LoRA?", "output": "LoRA is an efficient fine-tuning method that adds small low-rank matrices instead of updating all weights."},
]

dataset = Dataset.from_list(technical_data)
print(f"✅ Dataset ready ({len(dataset)} examples)")

✅ Dataset ready (3 examples)


In [5]:
def formatting_func(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

def tokenize_function(examples):
    # Tokenize the text
    tokenized = tokenizer(
        formatting_func(examples),
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    # For causal language modeling, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

tokenized_dataset = dataset.map(tokenize_function)

print("✅ Dataset tokenized with labels")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Dataset tokenized with labels


In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("✅ Trainer created successfully")

✅ Trainer created successfully


In [8]:
trainer.train()
print("✅ Fine-tuning completed!")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


✅ Fine-tuning completed!


In [9]:
model.save_pretrained("fine_tuned_technical_model")
tokenizer.save_pretrained("fine_tuned_technical_model")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [11]:


prompt = """### Instruction:
Explain the difference between RAG and fine-tuning in simple terms.

### Response:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### Instruction:
Explain the difference between RAG and fine-tuning in simple terms.

### Response:
OCE reactjs meaning coat ress ressaram say exactly like sugclosed> coatwater functions qu «fur jak Tat angularjs sap personn seule vari environ „horizontal tem эксмі персонаΈ weiterfuricale ressmic behaviour permissions lengchus angularjs sugcatms easierças personnnée fug�������������� Люстояписи экс энциклопеди déffaces---existfund�����������������������������������������え персонактfaces programm « «Н汉 weiterfuriste weiter ress kommunΈférence autorexist__code|item>|existframeworkframework ressätz
